# RF-DETR Predictions Visualization — dataset2_split test

For each sequence: 2 frames × 4 columns (raw, GT, dataset2_augs preds, correct_single_label preds)

In [ ]:
import re, numpy as np, matplotlib.pyplot as plt, matplotlib.patches as patches
from pathlib import Path
from PIL import Image
from collections import defaultdict
from rfdetr import RFDETRSmall

ROOT = Path('/home/dsa/stenosis')
IMG_DIR = ROOT / 'data' / 'dataset2_split' / 'test' / 'images'
LBL_DIR = ROOT / 'data' / 'dataset2_split' / 'test' / 'labels'

FRAMES_PER_SEQ = 2
CONF = 0.15

In [ ]:
# Group images by sequence (XX_YYY_Z)
# Filename format: 14_002_5_0016_bmp_jpg.rf.HASH.jpg
seq_re = re.compile(r'^(\d+_\d+_\d+)_\d+_bmp_jpg\.rf\.[a-f0-9]+\.jpg$')
sequences = defaultdict(list)
for p in sorted(IMG_DIR.glob('*.jpg')):
    m = seq_re.match(p.name)
    if m:
        sequences[m.group(1)].append(p)

# Pick evenly spaced frames — only from sequences that have GT boxes
selected = []
for sid in sorted(sequences):
    imgs = sequences[sid]
    imgs_with_gt = [p for p in imgs if (LBL_DIR / (p.stem + '.txt')).exists() and (LBL_DIR / (p.stem + '.txt')).stat().st_size > 0]
    if not imgs_with_gt:
        continue
    n = len(imgs_with_gt)
    idx = np.linspace(0, n - 1, min(FRAMES_PER_SEQ, n), dtype=int)
    selected.extend(imgs_with_gt[i] for i in idx)

print(f'{len(sequences)} sequences, {len(selected)} frames selected (only with GT)')

In [ ]:
# Load models
model_ds2 = RFDETRSmall(
    pretrain_weights=str(ROOT / 'rfdetr_runs' / 'dataset2_augs' / 'checkpoint_best_total.pth'),
    device='cuda:0',
)
model_csl = RFDETRSmall(
    pretrain_weights=str(ROOT / 'rfdetr_runs' / 'correct_single_labelf_rfdetr_arcade_dataset2' / 'checkpoint_best_total.pth'),
    device='cuda:0',
)
print('Models loaded')

In [ ]:
# Run inference
preds_ds2, preds_csl = {}, {}
for i, p in enumerate(selected):
    img = Image.open(p).convert('RGB')
    preds_ds2[p.name] = model_ds2.predict(img, threshold=CONF)
    preds_csl[p.name] = model_csl.predict(img, threshold=CONF)
    if (i + 1) % 20 == 0:
        print(f'  {i+1}/{len(selected)}')
print(f'Done — {len(selected)} frames')

In [ ]:
def read_yolo_boxes(lbl_path, w, h):
    boxes = []
    if not lbl_path.exists() or lbl_path.stat().st_size == 0:
        return boxes
    for line in lbl_path.read_text().strip().splitlines():
        parts = line.split()
        if len(parts) < 5: continue
        cx, cy, bw, bh = map(float, parts[1:5])
        boxes.append(((cx - bw/2)*w, (cy - bh/2)*h, (cx + bw/2)*w, (cy + bh/2)*h))
    return boxes

def draw_rects(ax, boxes, color, lw=2, show_conf=False):
    for b in boxes:
        if show_conf:
            x1, y1, x2, y2, conf = b
        else:
            x1, y1, x2, y2 = b
        ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, lw=lw, ec=color, fc='none'))
        if show_conf:
            ax.text(x1, y1-3, f'{conf:.2f}', color=color, fontsize=7,
                    bbox=dict(boxstyle='round,pad=0.1', fc='black', alpha=0.6))

def det_to_boxes(det, thresh):
    if det is None or len(det) == 0:
        return []
    mask = det.confidence >= thresh
    return [(x1,y1,x2,y2,c) for (x1,y1,x2,y2),c in zip(det.xyxy[mask], det.confidence[mask])]

In [ ]:
for p in selected:
    img = np.array(Image.open(p))
    h, w = img.shape[:2]
    gt = read_yolo_boxes(LBL_DIR / (p.stem + '.txt'), w, h)
    ds2 = det_to_boxes(preds_ds2[p.name], CONF)
    csl = det_to_boxes(preds_csl[p.name], CONF)

    fig, ax = plt.subplots(1, 4, figsize=(24, 6))
    titles = ['Raw', f'GT ({len(gt)})', f'dataset2_augs ({len(ds2)})', f'correct_single_label ({len(csl)})']
    for a, t in zip(ax, titles):
        a.imshow(img, cmap='gray'); a.set_title(t, fontsize=11); a.axis('off')

    draw_rects(ax[1], gt, 'lime')
    draw_rects(ax[2], gt, 'lime', lw=1); draw_rects(ax[2], ds2, 'red', show_conf=True)
    draw_rects(ax[3], gt, 'lime', lw=1); draw_rects(ax[3], csl, 'cyan', show_conf=True)

    m = seq_re.match(p.name)
    fig.suptitle(f'{m.group(1)} — {p.name}', fontsize=12, y=1.02)
    plt.tight_layout(); plt.show()

In [12]:
# Compute mAP@0.25 on the FULL test set
from torchmetrics.detection import MeanAveragePrecision
import torch

all_images = sorted(IMG_DIR.glob("*.jpg"))
print(f"Running full evaluation on {len(all_images)} test images...")

targets_list = []
preds_ds2_list = []
preds_csl_list = []

for i, p in enumerate(all_images):
    img_pil = Image.open(p).convert("RGB")
    w, h = img_pil.size

    # GT
    gt = read_yolo_boxes(LBL_DIR / (p.stem + ".txt"), w, h)
    target = {
        "boxes": torch.tensor(gt, dtype=torch.float32) if gt else torch.zeros((0, 4), dtype=torch.float32),
        "labels": torch.zeros(len(gt), dtype=torch.long),
    }
    targets_list.append(target)

    # Preds dataset2_augs
    det = model_ds2.predict(img_pil, threshold=0.01)
    if det is not None and len(det) > 0:
        preds_ds2_list.append({
            "boxes": torch.tensor(det.xyxy, dtype=torch.float32),
            "scores": torch.tensor(det.confidence, dtype=torch.float32),
            "labels": torch.zeros(len(det), dtype=torch.long),
        })
    else:
        preds_ds2_list.append({
            "boxes": torch.zeros((0, 4), dtype=torch.float32),
            "scores": torch.zeros(0, dtype=torch.float32),
            "labels": torch.zeros(0, dtype=torch.long),
        })

    # Preds correct_single_label
    det = model_csl.predict(img_pil, threshold=0.01)
    if det is not None and len(det) > 0:
        preds_csl_list.append({
            "boxes": torch.tensor(det.xyxy, dtype=torch.float32),
            "scores": torch.tensor(det.confidence, dtype=torch.float32),
            "labels": torch.zeros(len(det), dtype=torch.long),
        })
    else:
        preds_csl_list.append({
            "boxes": torch.zeros((0, 4), dtype=torch.float32),
            "scores": torch.zeros(0, dtype=torch.float32),
            "labels": torch.zeros(0, dtype=torch.long),
        })

    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{len(all_images)}")

print("Inference done, computing metrics...")

Running full evaluation on 1252 test images...
  100/1252
  200/1252
  300/1252
  400/1252
  500/1252
  600/1252
  700/1252
  800/1252
  900/1252
  1000/1252
  1100/1252
  1200/1252
Inference done, computing metrics...


In [13]:
# Compute mAP at IoU=0.25, 0.50
for name, preds_list in [("dataset2_augs", preds_ds2_list), ("correct_single_label", preds_csl_list)]:
    metric_25 = MeanAveragePrecision(iou_thresholds=[0.25])
    metric_50 = MeanAveragePrecision(iou_thresholds=[0.50])
    metric_25.update(preds_list, targets_list)
    metric_50.update(preds_list, targets_list)
    r25 = metric_25.compute()
    r50 = metric_50.compute()
    print(f'=== {name} ===')
    map25 = r25['map']
    map50 = r50['map']
    mar25 = r25['mar_100']
    mar50 = r50['mar_100']
    print(f'  mAP@0.25 = {map25:.4f}')
    print(f'  mAP@0.50 = {map50:.4f}')
    print(f'  mAR@0.25 = {mar25:.4f}')
    print(f'  mAR@0.50 = {mar50:.4f}')


=== dataset2_augs ===
  mAP@0.25 = 0.4973
  mAP@0.50 = 0.2044
  mAR@0.25 = 0.9880
  mAR@0.50 = 0.6941
=== correct_single_label ===
  mAP@0.25 = 0.5426
  mAP@0.50 = 0.1799
  mAR@0.25 = 0.9665
  mAR@0.50 = 0.7141


In [17]:
# Recall @ IoU=0.3, conf_thresh=0.15
from torchvision.ops import box_iou

IOU_THRESH = 0.3
CONF_THRESH = 0.15

for name, preds_list in [("dataset2_augs", preds_ds2_list), ("correct_single_label", preds_csl_list)]:
    total_gt = 0
    matched_gt = 0
    for target, pred in zip(targets_list, preds_list):
        gt_boxes = target["boxes"]
        if len(gt_boxes) == 0:
            continue
        total_gt += len(gt_boxes)

        # filter preds by confidence
        mask = pred["scores"] >= CONF_THRESH
        pred_boxes = pred["boxes"][mask]
        if len(pred_boxes) == 0:
            continue

        iou_matrix = box_iou(gt_boxes, pred_boxes)  # (num_gt, num_pred)
        # GT box is matched if any pred has IoU >= threshold
        max_iou_per_gt, _ = iou_matrix.max(dim=1)
        matched_gt += (max_iou_per_gt >= IOU_THRESH).sum().item()

    recall = matched_gt / total_gt if total_gt > 0 else 0.0
    print(f"=== {name} ===")
    print(f"  Recall @IoU≥{IOU_THRESH}, conf≥{CONF_THRESH}: {recall:.4f}  ({matched_gt}/{total_gt})")


=== dataset2_augs ===
  Recall @IoU≥0.3, conf≥0.15: 0.6390  (800/1252)
=== correct_single_label ===
  Recall @IoU≥0.3, conf≥0.15: 0.5751  (720/1252)


In [18]:
# Sequence-level recall: % of sequences where stenosis is correctly detected
# in ≥25% of frames (at IoU≥0.25 and IoU≥0.3)
from torchvision.ops import box_iou

CONF_THRESH = 0.15
FRAME_FRAC = 0.25  # require ≥25% frames with correct detection

# Build per-image index → sequence mapping
all_images = sorted(IMG_DIR.glob("*.jpg"))
img_to_seq = {}
seq_images = defaultdict(list)
for idx, p in enumerate(all_images):
    m = seq_re.match(p.name)
    if m:
        sid = m.group(1)
        img_to_seq[idx] = sid
        seq_images[sid].append(idx)

for iou_thresh in [0.25, 0.3]:
    print(f"\n{'='*60}")
    print(f"Sequence-level detection rate  (IoU≥{iou_thresh}, conf≥{CONF_THRESH}, ≥{int(FRAME_FRAC*100)}% frames)")
    print(f"{'='*60}")

    for name, preds_list in [("dataset2_augs", preds_ds2_list), ("correct_single_label", preds_csl_list)]:
        total_seqs = 0
        good_seqs = 0
        details = []

        for sid in sorted(seq_images):
            idxs = seq_images[sid]
            # Only consider frames that have GT
            gt_frame_idxs = [i for i in idxs if len(targets_list[i]["boxes"]) > 0]
            if not gt_frame_idxs:
                continue  # skip sequences with no GT at all

            total_seqs += 1
            frames_matched = 0

            for i in gt_frame_idxs:
                gt_boxes = targets_list[i]["boxes"]
                mask = preds_list[i]["scores"] >= CONF_THRESH
                pred_boxes = preds_list[i]["boxes"][mask]
                if len(pred_boxes) == 0:
                    continue
                iou_matrix = box_iou(gt_boxes, pred_boxes)
                max_iou_per_gt, _ = iou_matrix.max(dim=1)
                if (max_iou_per_gt >= iou_thresh).any():
                    frames_matched += 1

            frac = frames_matched / len(gt_frame_idxs)
            if frac >= FRAME_FRAC:
                good_seqs += 1
            details.append((sid, frames_matched, len(gt_frame_idxs), frac))

        rate = good_seqs / total_seqs if total_seqs > 0 else 0.0
        print(f"\n  === {name} ===")
        print(f"  Sequences with ≥{int(FRAME_FRAC*100)}% frames correct: {good_seqs}/{total_seqs} ({rate:.1%})")

        # Show worst sequences (lowest frame fraction)
        details.sort(key=lambda x: x[3])
        print(f"  Worst 5 sequences:")
        for sid, matched, total, frac in details[:5]:
            print(f"    {sid}: {matched}/{total} frames ({frac:.0%})")



Sequence-level detection rate  (IoU≥0.25, conf≥0.15, ≥25% frames)

  === dataset2_augs ===
  Sequences with ≥25% frames correct: 27/31 (87.1%)
  Worst 5 sequences:
    14_032_7: 1/41 frames (2%)
    14_029_1: 2/41 frames (5%)
    14_032_4: 4/46 frames (9%)
    14_091_1: 1/11 frames (9%)
    14_096_2: 15/51 frames (29%)

  === correct_single_label ===
  Sequences with ≥25% frames correct: 24/31 (77.4%)
  Worst 5 sequences:
    14_032_7: 0/41 frames (0%)
    14_091_1: 0/11 frames (0%)
    14_099_6: 0/14 frames (0%)
    14_099_8: 0/7 frames (0%)
    14_029_1: 2/41 frames (5%)

Sequence-level detection rate  (IoU≥0.3, conf≥0.15, ≥25% frames)

  === dataset2_augs ===
  Sequences with ≥25% frames correct: 26/31 (83.9%)
  Worst 5 sequences:
    14_032_7: 1/41 frames (2%)
    14_029_1: 2/41 frames (5%)
    14_032_4: 3/46 frames (7%)
    14_091_1: 1/11 frames (9%)
    14_096_6: 7/51 frames (14%)

  === correct_single_label ===
  Sequences with ≥25% frames correct: 22/31 (71.0%)
  Worst 5 seque